# SkyLedger Data Quality Gate

Critical data-quality checks across bronze, silver, and gold. The notebook **fails the pipeline** (raises) when any critical rule is violated, so a run stops instead of publishing bad data. Covers: bronze ingestion completeness for all 8 source tables, primary keys populated and unique, fact-table surrogate keys and business rules (ticket price > 0, discount within [0,1], seats >= 1), gold-to-silver reconciliation of row counts and revenue, and rejected-record reporting per table.

In [0]:
dbutils.widgets.text("catalog", "sl")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

# Every critical check registers its outcome here. The gate cell at the end
# raises (and fails the pipeline / job task) if any entry has status FAIL.
dq_results = []

def run_check(check_id, category, description, violations, details=""):
    """Register a critical check. violations == 0 -> PASS, otherwise FAIL."""
    status = "PASS" if int(violations) == 0 else "FAIL"
    dq_results.append(
        Row(check_id=check_id, category=category, description=description,
            violations=int(violations), status=status, details=str(details))
    )
    print(f"[{status}] {check_id} | {description} | {details}")
    return status

In [0]:
# --- 1. Bronze ingestion: all 8 source tables present and non-empty ---
bronze_tables = ["region", "nation", "airline", "route",
                 "airline_route", "passenger", "booking", "flight_booking"]

missing = [t for t in bronze_tables
           if not spark.catalog.tableExists(f"{catalog}.bronze.{t}")]
run_check("bronze.all_tables_present", "BRONZE_INGESTION",
          "All 8 source tables ingested into bronze", len(missing),
          f"missing={missing or 'none'}")

empty_tables = [t for t in bronze_tables
                if t not in missing
                and spark.table(f"{catalog}.bronze.{t}").limit(1).count() == 0]
run_check("bronze.all_tables_non_empty", "BRONZE_INGESTION",
          "Every bronze table contains at least one row", len(empty_tables),
          f"empty={empty_tables or 'none'}")

In [0]:
# --- 2. Primary keys populated and unique (silver layer) ---
pk_specs = {
    "region":         ["r_regionkey"],
    "nation":         ["n_nationkey"],
    "airline":        ["al_airlineid"],
    "route":          ["ro_routeid"],
    "airline_route":  ["ar_airlineid", "ar_routeid"],     # composite PK
    "passengers":     ["pa_passengerid"],
    "booking":        ["bk_bookingid"],
    "flight_booking": ["fb_bookingid", "fb_linenumber"],   # composite PK
}

for table, keys in pk_specs.items():
    df = spark.table(f"{catalog}.silver.{table}")
    total = df.count()

    null_cond = " OR ".join(f"{k} IS NULL" for k in keys)
    null_rows = df.where(null_cond).count()
    run_check(f"pk.{table}.populated", "PRIMARY_KEY",
              f"silver.{table} PK {keys} fully populated", null_rows,
              f"null_rows={null_rows} of {total}")

    distinct_pk = df.dropna(subset=keys).select(*keys).distinct().count()
    dup_rows = (total - null_rows) - distinct_pk
    run_check(f"pk.{table}.unique", "PRIMARY_KEY",
              f"silver.{table} PK {keys} unique", dup_rows,
              f"duplicate_rows={dup_rows} of {total}")

In [0]:
# --- 3. Fact table: no null surrogate keys + business rules on measures ---
fb = spark.table(f"{catalog}.silver.flight_booking")

null_surr = fb.where(
    "fb_bookingid IS NULL OR fb_routeid IS NULL OR fb_airlineid IS NULL"
).count()
run_check("fact.null_surrogate_keys", "FACT_INTEGRITY",
          "flight_booking has no null surrogate keys (bookingid, routeid, airlineid)",
          null_surr)

bad_price = fb.where("fb_ticketprice IS NULL OR fb_ticketprice <= 0").count()
run_check("fact.ticket_price_positive", "BUSINESS_RULE",
          "fb_ticketprice > 0 and not null", bad_price)

bad_discount = fb.where(
    "fb_discount IS NULL OR fb_discount < 0 OR fb_discount > 1"
).count()
run_check("fact.discount_in_0_1", "BUSINESS_RULE",
          "fb_discount within [0, 1] inclusive", bad_discount)

bad_seats = fb.where("fb_seats IS NULL OR fb_seats < 1").count()
run_check("fact.seats_at_least_1", "BUSINESS_RULE",
          "fb_seats >= 1 and not null", bad_seats)

In [0]:
# --- 4. Gold reconciliation: row counts and revenue vs silver ---
for gold_table, silver_table in [
    ("dim_passenger", "passengers"),
    ("dim_route", "route"),
    ("dim_airline", "airline"),
    ("dim_geography", "nation"),
]:
    g = spark.table(f"{catalog}.gold.{gold_table}").count()
    s = spark.table(f"{catalog}.silver.{silver_table}").count()
    run_check(f"gold.rowcount.{gold_table}", "GOLD_RECON",
              f"gold.{gold_table} row count reconciles with silver.{silver_table}",
              abs(g - s), f"gold={g} silver={s}")

silver_revenue = spark.table(f"{catalog}.silver.booking") \
    .agg(F.sum("bk_totalprice").alias("rev")).collect()[0]["rev"] or 0
gold_revenue = spark.sql(
    f"SELECT COALESCE(SUM(annual_revenue), 0) AS rev FROM {catalog}.gold.vw_annual_booking_kpi"
).collect()[0]["rev"]
rev_diff = abs(float(silver_revenue) - float(gold_revenue))
run_check("gold.revenue.booking_kpi", "GOLD_RECON",
          "vw_annual_booking_kpi total revenue reconciles with silver.booking",
          0 if rev_diff <= 0.01 else 1,
          f"silver={silver_revenue} gold={gold_revenue} diff={rev_diff:.2f}")

gold_bookings = spark.sql(
    f"SELECT COALESCE(SUM(distinct_bookings), 0) AS b FROM {catalog}.gold.vw_annual_booking_kpi"
).collect()[0]["b"]
silver_bookings = spark.table(f"{catalog}.silver.booking") \
    .select("bk_bookingid").distinct().count()
run_check("gold.bookings.booking_kpi", "GOLD_RECON",
          "vw_annual_booking_kpi distinct bookings reconcile with silver.booking",
          abs(gold_bookings - silver_bookings),
          f"gold={gold_bookings} silver={silver_bookings}")

In [0]:
# --- 5. Rejected records: reported per table + bronze completeness ---
# bronze rows must equal silver valid rows + quarantined (rejected) rows
silver_names = {
    "region": "region", "nation": "nation", "airline": "airline",
    "route": "route", "airline_route": "airline_route",
    "passenger": "passengers", "booking": "booking",
    "flight_booking": "flight_booking",
}

reject_rows = []
for bronze_t, silver_t in silver_names.items():
    bronze_cnt = spark.table(f"{catalog}.bronze.{bronze_t}").count()
    silver_cnt = spark.table(f"{catalog}.silver.{silver_t}").count()
    quarantine_cnt = spark.table(f"{catalog}.silver.{silver_t}_quarantine").count()

    run_check(f"reject.{silver_t}.reconciles", "REJECT_REPORT",
              f"bronze.{bronze_t} rows == silver.{silver_t} valid + quarantine",
              abs(bronze_cnt - silver_cnt - quarantine_cnt),
              f"bronze={bronze_cnt} valid={silver_cnt} rejected={quarantine_cnt}")

    reject_rows.append(Row(table=bronze_t, bronze_rows=bronze_cnt,
                           silver_valid_rows=silver_cnt, rejected_rows=quarantine_cnt))

print("Rejected (quarantined) records per source table:")
display(spark.createDataFrame(reject_rows))

In [0]:
# --- Gate: fail the pipeline if any critical check is violated ---
dq_df = spark.createDataFrame(dq_results)
display(dq_df)

failed = [r.check_id for r in dq_results if r.status == "FAIL"]
print(f"Checks executed: {len(dq_results)} | passed: {len(dq_results) - len(failed)} | failed: {len(failed)}")

if failed:
    raise Exception(
        f"DATA QUALITY GATE FAILED — {len(failed)} critical check(s) violated: "
        + ", ".join(failed)
    )

print("DATA QUALITY GATE PASSED — all critical checks satisfied")